[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/HannahPinson/tue-deeplearning-2AMM10/tree/main/tutorials/P3.1_huggingface_intro.ipynb)

# Getting started with the Hugging Face ecosystem 🤗

This tutorial introduces the parts of Hugging Face that are most useful when you want to **use**, **inspect**, and eventually **fine-tune** deep learning models.
By the end of this notebook, you should be able to:

1. use `pipeline(...)` for quick inference.
2. choose a model from the Hugging Face Hub.
3. understand the difference between high-level pipelines and lower-level model APIs.
4. inspect model inputs and outputs.
5. load and explore datasets with the `datasets` library.
6. understand how this workflow connects to later assignments on evaluating and fine-tuning segmentation models.


## 0. Setup

The main libraries used in this notebook are:

- `transformers`: models, processors, tokenizers, and pipelines;
- `datasets`: loading, inspecting, and preprocessing datasets;
- `torch`: tensor operations and model execution;
- `PIL` and `matplotlib`: simple image loading and visualization.

If you are running this notebook in Google Colab, (you may need to) uncomment and run the installation cell below.


In [ ]:
# If needed, uncomment this cell in Colab:
# !pip install -q transformers datasets accelerate evaluate pillow matplotlib torchvision

In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
import requests
import torch
from PIL import Image
from torchvision.transforms import functional as F
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    pipeline,
)
from transformers.utils import logging

# Keep notebook output readable (HF Logs).
logging.set_verbosity_error()

DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else (torch.device("mps") if torch.mps.is_available() else torch.device("cpu"))
)
print(f"Using device: {DEVICE}")

## 1. What is Hugging Face?

Hugging Face is an ecosystem for sharing and using machine learning resources. For this tutorial, the most important parts are:

- the **Hub**, where people publish models, datasets, and demos;
- the **`transformers`** library, which provides many pre-trained models and convenient APIs;
- the **`datasets`** library, which provides a standard interface for loading and preprocessing datasets;
- model and dataset cards, which document what a resource was trained on, how it should be used, and what its limitations are.

A useful mental model is: **Hugging Face is like GitHub for machine learning artifacts, with Python libraries that make those artifacts easy to use.**

In this tutorial we first use high-level APIs, then gradually look underneath the abstraction.


## 2. The `pipeline(...)` abstraction

The `pipeline()` function is the quickest way to run inference with a pre-trained model.

A pipeline usually handles four steps for you:

```text
raw input → preprocessing → model forward pass → postprocessing → readable output
```

This is helpful when you want to try a model quickly, demonstrate a task, or verify that your environment is working.

However, pipelines hide many details. Later in the notebook, we will open the black box and use the processor and model directly.


### 2.1 Example: sentiment analysis

Sentiment analysis is a text classification task. Given a sentence, the model predicts whether the sentiment is positive, negative, or sometimes neutral, depending on the model.

Here we create a sentiment-analysis pipeline:

In [ ]:
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=DEVICE,
)

input_sentence = "The 2AMM10 Deep Learning course is awesome! I am loving it so far."
sentiment_pipeline(input_sentence)

The output is a list of dictionaries. Each dictionary contains:

- `label`: the predicted class;
- `score`: the model's confidence for that predicted class.



#### Try your own

Try a few sentences of your own. Include at least one sentence where the sentiment is ambiguous context-specific or sarcastic.

What happens to the score? Does the predicted label match your own interpretation?


In [ ]:
my_sentences = [
    "I'm gonna make him an offer he can't refuse.",
    "Here’s Johnny!",
    "Is this the real life? Is this just fantasy? Caught in a landslide, no escape from reality",
]

sentiment_pipeline(my_sentences)

## 3. Understanding what `pipeline()` hides

So far, the interface has been simple:

```python
pipeline(task=..., model=...)(input)
```

This works across different tasks and modalities, which is one reason the Hugging Face API is popular.

But for research and assignments, you usually need more control. For example, you may need to:

- choose a specific checkpoint;
- inspect the preprocessing step;
- access raw logits;
- compute your own metrics;
- fine-tune the model;
- save or upload a trained checkpoint.

For that, we need to understand the lower-level API.


## 4. Choosing models from the Hugging Face Hub

In practice, you should not choose models blindly. A good workflow is:

1. go to the [Hugging Face Hub](https://huggingface.co/models );
2. filter by task, such as `image-classification`, `semantic-segmentation`, or `text-classification`;
3. inspect the model card (think of it like the model's README file);
4. check the intended use, training data, license, limitations, and example code;
5. load the model and the matching pre-processor in Python.

For image models, the **image processor** is especially important. It defines transformations such as resizing, cropping, normalization, and conversion to tensors.

In the next example, we use the image-classification model `microsoft/resnet-18`.


In [ ]:
model_name = "microsoft/resnet-18"

image_processor = AutoImageProcessor.from_pretrained(
    model_name
)  # preprocessing pipeline for the model
model = AutoModelForImageClassification.from_pretrained(model_name).to(
    DEVICE
)  # load the model and move it to the appropriate device
model.eval()

Let us inspect the image before and after preprocessing.


In [ ]:
# Get Image
image_url = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTg11i7a72Um3lG94xW3PaB_udrp3gOBhu7Zw&s"
response = requests.get(image_url, timeout=10)
response.raise_for_status()
image = Image.open(BytesIO(response.content)).convert("RGB")

processed = image_processor(image, return_tensors="pt")
processed_image = processed["pixel_values"][0]


fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].imshow(image)
ax[0].set_title("Original image")
ax[0].axis("off")

ax[1].imshow(F.to_pil_image(processed_image))
ax[1].set_title("After preprocessing")
ax[1].axis("off")

plt.show()

The processed image may look different because the processor has resized and normalized it for the model.

The shape of `pixel_values` is usually:

```text
batch_size × channels × height × width
```

For PyTorch image models, this is often called `BCHW` format.


## 5. Running the model manually

Now we run the image through the model without using a pipeline object.

This gives us access to the raw output logits. A logit is an unnormalized score. To get probabilities, we apply `softmax`.


In [ ]:
inputs = image_processor(image, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
probabilities = torch.softmax(logits, dim=-1)

predicted_class_id = probabilities.argmax(dim=-1).item()
predicted_label = model.config.id2label[predicted_class_id]
predicted_score = probabilities[0, predicted_class_id].item()

print(f"Predicted class: {predicted_label}")
print(f"Confidence: {predicted_score:.3f}")
print("Logits shape:", logits.shape)

The logits shape tells us how many output classes the classifier has. For ImageNet-style classifiers, this is often 1000 classes.

To inspect more than just the top prediction, we can compute the top-*k* classes ourselves.


In [ ]:
top_k = 5
top_probs, top_ids = torch.topk(probabilities[0], k=top_k)

for rank, (class_id, score) in enumerate(
    zip(top_ids.tolist(), top_probs.tolist()), start=1
):
    print(f"{rank}. {model.config.id2label[class_id]:30s} {score:.3f}")

### 5.1 Comparing with the pipeline

The pipeline performs the same broad steps, but wraps them into a compact interface:

1. preprocess the image;
2. run the model;
3. apply postprocessing;
4. return the top labels and scores.


In [ ]:
image_classification_pipeline = pipeline(
    task="image-classification",
    model=model_name,
    device=DEVICE,
)

image_classification_pipeline(image)

The pipeline is convenient for inference. The lower-level API is better when you need to train, evaluate, debug, or customize the workflow.

A useful rule of thumb:

- use **pipelines** when you want quick predictions;
- use **processors/tokenizers + models** when you need control (But with great power comes great responsibility!).


## 6. Hugging Face Datasets

The `datasets` library gives a standard way to load, inspect, transform, and share datasets.

The basic workflow is:

1. load a dataset;
2. inspect its splits and features;
3. look at individual examples;
4. preprocess examples into model-ready tensors;
5. pass the processed dataset to a training or evaluation loop.


### 6.1 Loading a dataset from the Hub

The function `load_dataset(...)` downloads and prepares a dataset. Many datasets are organized as a `DatasetDict`, which contains splits such as `train`, `validation`, and `test`.

Here we use a small image classification dataset for demonstration.


In [ ]:
from datasets import load_dataset

# A small dataset that is convenient for a quick demonstration.
dataset = load_dataset("beans")

dataset

### 6.2 Inspecting splits, features, and examples

Before training any model, always inspect the dataset.

Useful questions are:

- Which splits are available?
- What fields does each example contain?
- Are images decoded as PIL images?
- What are the label names?
- Are the train/validation/test splits balanced?


In [ ]:
print(dataset.keys())
print(dataset["train"].features)
print(dataset["train"][0])

In [ ]:
example = dataset["train"][0]
print("example is a dictionary with keys:", example.keys())
plt.imshow(example["image"])
plt.title(f"Label: {dataset['train'].features['labels'].names[example['labels']]}")
plt.axis("off")
plt.show()

### 6.3 Selecting and slicing data

During debugging, it is often useful to work with a small subset before running a full experiment.


In [ ]:
small_train_dataset = dataset["train"].select(range(8))
print(small_train_dataset)

### 6.4 Applying transformations lazily with `with_transform`

For image tasks, we often want to apply preprocessing only when examples are read. This avoids storing a second copy of the full preprocessed dataset on disk.

The method `with_transform(...)` lets us define a transformation that is applied lazily.


In [ ]:
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-18")


def preprocess_batch(batch):
    # batch["image"] is a list of PIL images.
    inputs = processor(batch["image"], return_tensors="pt")
    inputs["labels"] = torch.tensor(batch["labels"])
    return inputs


prepared_train = dataset["train"].with_transform(preprocess_batch)

batch = prepared_train[:4]
print(batch.keys())
print(batch["pixel_values"].shape)
print(batch["labels"])

This gives us model-ready tensors while keeping the original dataset clean.

For a real training loop, you would usually combine this with a `DataLoader` or the Hugging Face `Trainer` API.


### 6.6 Dataset cards and responsible use

Just like models, datasets on the Hub usually have dataset cards. You should read them before using a dataset.

A good dataset card helps you answer:

- What does the dataset contain?
- How was it collected and annotated?
- What are the labels?
- What are the intended uses and limitations?


## 7 Summary

In this tutorial, you have seen three levels of the Hugging Face workflow:

1. **Pipelines** for quick inference;
2. **processors + models** for lower-level control;
3. **datasets** for loading and preprocessing data in a reproducible way.

The complete HF documentation can be found [here](https://huggingface.co/docs).
